In [4]:
!pip install chardet


[notice] A new release of pip is available: 24.3.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import random
import chardet

def detect_encoding(file_path):
    """
    Automatically detect the encoding of a file
    
    Parameters:
    file_path (str): Path to the file
    
    Returns:
    str: Detected encoding
    """
    # Read a sample of the file to detect encoding
    with open(file_path, 'rb') as f:
        # Read up to 100KB to detect encoding
        raw_data = f.read(100000)
    
    # Detect encoding
    result = chardet.detect(raw_data)
    encoding = result['encoding']
    confidence = result['confidence']
    
    print(f"Detected encoding: {encoding} with confidence: {confidence}")
    return encoding

def create_balanced_dataset(input_file, output_file, samples_per_label=200, text_column='text', label_column='sentiment', 
                           encoding=None, error_handling='replace'):
    """
    Create a balanced dataset with an equal number of samples for each sentiment label.
    
    Parameters:
    input_file (str): Path to the input CSV file
    output_file (str): Path to save the balanced CSV file
    samples_per_label (int): Number of samples to select for each label
    text_column (str): Name of the column containing the text
    label_column (str): Name of the column containing the sentiment labels
    encoding (str): File encoding (if None, will try to auto-detect)
    error_handling (str): How to handle encoding errors: 'strict', 'ignore', or 'replace'
    
    Returns:
    pd.DataFrame: The balanced dataset
    """
    # List of encodings to try if auto-detection fails
    encoding_options = ["utf-8", "latin1", "ISO-8859-1", "cp1252", "utf-16", "utf-32"]
    
    # Load the dataset
    try:
        # Auto-detect encoding if not provided
        if encoding is None:
            try:
                encoding = detect_encoding(input_file)
            except Exception as e:
                print(f"Error detecting encoding: {e}")
                print("Will try multiple encodings...")
        
        # Try loading with detected or provided encoding
        try:
            df = pd.read_csv(input_file, encoding=encoding, encoding_errors=error_handling)
            print(f"Successfully loaded {len(df)} rows from {input_file} using {encoding} encoding")
        except Exception as enc_error:
            # If that fails, try various encodings
            print(f"Error with {encoding} encoding: {enc_error}")
            success = False
            
            for enc in encoding_options:
                try:
                    print(f"Trying {enc} encoding...")
                    df = pd.read_csv(input_file, encoding=enc, encoding_errors=error_handling)
                    print(f"Successfully loaded file with {enc} encoding")
                    success = True
                    break
                except Exception as e:
                    print(f"Failed with {enc} encoding: {e}")
            
            if not success:
                raise Exception("Could not load file with any encoding")
    
    except Exception as e:
        print(f"Error loading file: {e}")
        return None
    
    # Check if the required columns exist
    if text_column not in df.columns:
        print(f"Error: Text column '{text_column}' not found in the dataset.")
        print(f"Available columns: {df.columns.tolist()}")
        return None
    
    if label_column not in df.columns:
        print(f"Error: Label column '{label_column}' not found in the dataset.")
        print(f"Available columns: {df.columns.tolist()}")
        return None
    
    # Standardize sentiment labels (convert to lowercase and handle variations)
    df['standardized_sentiment'] = df[label_column].astype(str).str.lower()
    
    # Map different formats to standard format
    sentiment_mapping = {
        'positive': 'positive', 'pos': 'positive', '1': 'positive', '1.0': 'positive',
        'negative': 'negative', 'neg': 'negative', '0': 'negative', '0.0': 'negative', '-1': 'negative', '-1.0': 'negative',
        'neutral': 'neutral', 'neu': 'neutral', '2': 'neutral', '2.0': 'neutral'
    }
    
    # Apply the mapping
    df['standardized_sentiment'] = df['standardized_sentiment'].map(
        lambda x: next((sentiment_mapping[k] for k in sentiment_mapping if k == x), 'neutral')
    )
    
    # Get unique standardized labels
    unique_labels = df['standardized_sentiment'].unique()
    print(f"Found {len(unique_labels)} unique labels: {unique_labels}")
    
    # Create balanced dataset
    balanced_data = []
    
    for label in ['positive', 'neutral', 'negative']:
        if label in unique_labels:
            # Get all samples for this label
            label_samples = df[df['standardized_sentiment'] == label]
            
            if len(label_samples) >= samples_per_label:
                # If we have enough samples, randomly select the required number
                selected_samples = label_samples.sample(samples_per_label, random_state=42)
            else:
                # If we don't have enough samples, take all available and warn
                selected_samples = label_samples
                print(f"Warning: Only {len(label_samples)} samples available for '{label}' (requested {samples_per_label})")
            
            # Add to our balanced dataset
            balanced_data.append(selected_samples)
            print(f"Added {len(selected_samples)} samples for label '{label}'")
        else:
            print(f"Warning: No samples found for label '{label}'")
    
    # Check if we have any data to combine
    if not balanced_data:
        print("Error: No valid sentiment data found to create balanced dataset")
        return None
    
    # Combine all selected samples
    balanced_df = pd.concat(balanced_data, ignore_index=True)
    
    # Keep only the original columns plus the standardized sentiment
    if 'standardized_sentiment' not in df.columns:
        balanced_df = balanced_df[[text_column, label_column, 'standardized_sentiment']]
    else:
        balanced_df = balanced_df[[text_column, label_column]]
    
    # Shuffle the dataset
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Save to CSV with UTF-8 encoding
    try:
        balanced_df.to_csv(output_file, index=False, encoding='utf-8')
        print(f"Saved balanced dataset with {len(balanced_df)} rows to {output_file}")
    except Exception as e:
        print(f"Error saving file: {e}")
        # Try with a different encoding if UTF-8 fails
        try:
            balanced_df.to_csv(output_file, index=False, encoding='latin1')
            print(f"Saved balanced dataset with latin1 encoding")
        except Exception as e2:
            print(f"Error saving with latin1 encoding: {e2}")
    
    return balanced_df

# Example usage
if __name__ == "__main__":
    # Replace these with your actual file paths
    input_file = "all-data.csv"  
    output_file = "balanced_dataset.csv"
    
    # Create balanced dataset with 200 samples per label
    balanced_df = create_balanced_dataset(
        input_file=input_file,
        output_file=output_file,
        samples_per_label=200,
        text_column='text',  # Replace with your text column name
        label_column='sentiment',  # Replace with your sentiment column name
        encoding=None,  # Will auto-detect
        error_handling='replace'  # 'replace' will substitute problematic characters
    )
    
    # Print distribution of labels in the balanced dataset
    if balanced_df is not None:
        print("\nLabel distribution in balanced dataset:")
        print(balanced_df['standardized_sentiment'].value_counts())

Detected encoding: Windows-1252 with confidence: 0.73
Successfully loaded 4846 rows from all-data.csv using Windows-1252 encoding
Found 3 unique labels: ['neutral' 'negative' 'positive']
Added 200 samples for label 'positive'
Added 200 samples for label 'neutral'
Added 200 samples for label 'negative'
Saved balanced dataset with 600 rows to balanced_dataset.csv

Label distribution in balanced dataset:


KeyError: 'standardized_sentiment'